# JEPA Mini Project Colab Runner

This notebook installs dependencies, trains the JEPA and reconstruction baselines, runs downstream evaluations, exports latents, and displays the main embedding plots.

In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path('/content/jepa_mini_project')
if PROJECT_ROOT.exists():
    os.chdir(PROJECT_ROOT)
print('Working directory:', Path.cwd())

In [ ]:
!pip install -r requirements.txt

In [ ]:
import torch

if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print('Detected device:', device)

In [ ]:
!python -m src.train_jepa --config configs/colab.yaml

In [ ]:
!python -m src.train_mae --config configs/colab.yaml

In [ ]:
from pathlib import Path

run_dirs = sorted(Path('runs').glob('*'), key=lambda path: path.stat().st_mtime)
jepa_ckpt = None
mae_ckpt = None
for run_dir in reversed(run_dirs):
    if 'jepa' in run_dir.name and jepa_ckpt is None:
        jepa_ckpt = run_dir / 'checkpoints' / 'best.ckpt'
    if 'mae' in run_dir.name and mae_ckpt is None:
        mae_ckpt = run_dir / 'checkpoints' / 'best.ckpt'
print('JEPA checkpoint:', jepa_ckpt)
print('MAE checkpoint:', mae_ckpt)

In [ ]:
!python -m src.eval_linear_probe --checkpoint "$jepa_ckpt" --config configs/colab.yaml
!python -m src.eval_linear_probe --checkpoint "$mae_ckpt" --config configs/colab.yaml
!python -m src.eval_retrieval --checkpoint "$jepa_ckpt" --config configs/colab.yaml
!python -m src.eval_retrieval --checkpoint "$mae_ckpt" --config configs/colab.yaml
!python -m src.export_latents --checkpoint "$jepa_ckpt" --config configs/colab.yaml --split test
!python -m src.export_latents --checkpoint "$mae_ckpt" --config configs/colab.yaml --split test
!python -m src.visualize_embeddings --checkpoint "$jepa_ckpt" --checkpoint "$mae_ckpt" --config configs/colab.yaml

In [ ]:
from IPython.display import Image, display
from pathlib import Path

viz_candidates = sorted(Path('runs').glob('*/embedding_viz/embedding_comparison.png'))
if viz_candidates:
    display(Image(filename=str(viz_candidates[-1])))
else:
    print('No embedding plot found yet.')